# 03 · Span과 단계별 실행 추적

목표: 부모·자식 관계, 실행 시간, 동시 요청의 context 격리를 확인한다. 약15분.

**이 파일의 핵심 질문:** 어느 단계가 실패하거나 느렸으며, 어떤 Trace·Span 관계가 그 판단을 뒷받침하는가?

**Trace**는 한 요청의 실행 흐름, **Span**은 그 안의 작업 구간이다. **parent_id**는 부모 Span 연결 값, **ContextVar**는 동시에 실행되는 작업별 문맥을 분리하는 Python 도구다. **OpenTelemetry(OTel)**는 계측 데이터의 생성·수집·전달을 위한 도구 모음이다. 이 Notebook은 직접 구현한 Span을 사용하고 OTel SDK·Collector를 설치하지 않는다. **W3C**는 World Wide Web Consortium, 웹 표준화 기구다. **OTLP**는 OpenTelemetry Protocol, 계측 데이터 전송 규약이다.

**수업 중 먼저 실행할 단계**

- 1 정상 경로
- 2 부모·자식 검증
- 3 시간 해석
- 7 실패 지점

파일 안의 변수 의존성이 있으므로 처음 실행할 때는 위에서부터 순서대로 진행한다. 핵심 단계에서는 출력을 읽고, 나머지 단계는 빠르게 실행한 뒤 복습 때 살펴본다.

## 시작 전 · 어디에서 무엇을 실행하나요?

1. VS Code에서 `llmops-16week` **폴더 전체**를 연다. 압축을 풀었다면 Notebook 파일만 따로 옮기지 않는다.
2. VS Code의 Python·Jupyter 확장을 설치하고 우측 위 **Select Kernel → Python Environments → .venv / Python 3.11**을 선택한다. 터미널의 가상환경과 Notebook 커널은 별도 선택이다.
3. 아래 **환경 확인 코드 셀부터** `Shift+Enter`로 한 셀씩 실행한다. 이 파일 안에서는 앞에서 만든 변수를 뒤에서 사용한다.
4. 각 단계의 **예상 결과**와 실제 출력을 대조한다. `assert`는 확인 조건이다. 출력이나 오류가 없어도 셀이 완료되면 조건을 통과한 것이다.
5. 셀을 건너뛰어 `NameError`가 나거나 변수를 여러 번 바꿨다면 **Restart Kernel → Run All**로 처음부터 실행한다. 다른 Notebook의 변수는 가져오지 않는다.

처음 환경을 만드는 경우에만 **VS Code 터미널**에서 실행한다. 아래 명령을 Python 코드 셀에 붙여넣지 않는다.

```bash
# macOS / Linux · 저장소 루트
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -r requirements.txt
```

Windows PowerShell은 `py -3.11 -m venv .venv`, `.venv\Scripts\Activate.ps1` 순서다. 활성화가 제한된 환경에서는 `.venv\Scripts\python.exe -m pip install -r requirements.txt`처럼 가상환경 Python을 직접 실행한다. 이미 준비된 `.venv`는 그대로 쓴다.

**공통 실습의 실행 방식:** `lab_client`가 메모리 안에 FastAPI 앱을 만들고 시작·종료를 관리한다. HTTPX의 ASGITransport가 같은 프로세스에서 요청을 전달하므로 `uvicorn`, Docker, Ollama, API 키가 필요하지 않다. ASGI는 *Asynchronous Server Gateway Interface*, Python 비동기 웹 앱과 서버 사이의 인터페이스다.

Notebook 앱과 브라우저의 `127.0.0.1:8010` 앱은 **서로 다른 실행**이다. Notebook에서 만든 요청 수가 브라우저·Prometheus에 합쳐지지 않는다. 실제 모델 대신 고정 문장과 대기를 쓰므로 외부 비용은 발생하지 않는다. 실행 시간·요청 ID는 매번 달라져도 정상이다.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week10/lab/app.py").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import asyncio, json
from dataclasses import replace
from week10.lab.config import Settings
from week10.lab.experiments import lab_client, fixture_check, window_delta, compare_releases
assert sys.version_info[:2] == (3, 11), "Select the Python 3.11 .venv kernel in VS Code"
print("Python:", sys.version.split()[0])
print("실행 환경:", sys.executable)
print("저장소:", ROOT)
print("환경 확인 완료 · 이 Notebook은 별도 서버 없이 실행합니다.")

## 1. 정상 경로

검색·생성·출력을 assist 부모 작업으로 묶는다.

**예상 결과:** assist·retrieve·generate·format의4개 Span과 offset_ms·duration_ms.

**결과 해석:** assist가 전체 요청을 감싸고 나머지는 하위 작업이다. 보통 generate가 가장 오래 걸리지만 기기별 시간은 달라진다.

In [ ]:
async with lab_client() as (app, client):
    r = await client.post("/api/v1/assist", json={"message":"신청 절차"})
    snap = app.state.telemetry.snapshot()
spans = sorted(snap["spans"], key=lambda s:s["offset_ms"])
[(s["name"], s["offset_ms"], s["duration_ms"]) for s in spans]

## 2. 부모·자식 검증

자식 span은 assist의 span_id를 parent_id로 가진다.

**예상 결과:** 자식3개의 parent_id가 assist의 span_id와 같고 trace_id도 같다.

**결과 해석:** 이름이 비슷한 작업을 임의로 묶는 대신 ID 관계로 같은 요청의 실행임을 확인한다.

In [ ]:
root = next(s for s in spans if s["name"] == "assist")
children = [s for s in spans if s["name"] != "assist"]
assert all(s["parent_id"] == root["span_id"] for s in children)
assert all(s["trace_id"] == root["trace_id"] for s in children)

## 3. 시간 해석

부모 시간은 자식을 포함한다. 부모+자식 합계는 이중 집계다.

**예상 결과:** 부모 기간·자식 합계·가장 긴 자식 작업 이름.

**결과 해석:** 부모가 자식을 포함한다. 둘을 더하면 같은 시간을 중복 계산한다. 외부 네트워크·GPU 시간은 이 실험에 없다.

In [ ]:
print("부모 실행 ms:", root["duration_ms"])
print("자식 합계 ms:", sum(s["duration_ms"] for s in children))
print("가장 긴 자식:", max(children, key=lambda s:s["duration_ms"])["name"])

## 4. W3C header의 제한된 예시

00 버전·01 flags의 유효한 context만 전달한다. 전체 표준·샘플러·OTLP 구현이 아니다.

**예상 결과:** 전달한 a32자리 trace_id와 b16자리 원격 parent_id를 유지한다.

**결과 해석:** 헤더로 이전 작업과 연결하는 제한된 예시다. 전체 W3C 처리나 OpenTelemetry SDK가 구현된 것은 아니다.

In [ ]:
header = "00-" + "a"*32 + "-" + "b"*16 + "-01"
async with lab_client() as (app, client):
    r = await client.post("/api/v1/assist", json={"message":"예시"}, headers={"traceparent":header})
    root = next(s for s in app.state.telemetry.spans if s["name"] == "assist")
assert root["trace_id"] == "a"*32 and root["parent_id"] == "b"*16

## 5. 잘못된 context

올제로 ID나 지원하지 않는 버전은 새 ID로 시작한다.

**예상 결과:** 새 trace_id 두 개, parent=None. 잘못된 헤더는 예외 없이 새 문맥으로 시작한다.

**결과 해석:** 형식이 맞지 않는 ID를 그대로 신뢰하지 않는다. 이 교육용 파서는 지원 범위가 제한적이다.

In [ ]:
from week10.lab.telemetry import trace_context
for h in ["not-a-header", "00-"+"0"*32+"-"+"b"*16+"-01"]:
    tid, parent = trace_context(h)
    assert int(tid,16)>0 and parent is None
    print(tid)

## 6. 동시 요청 격리

한 context 전역 변수를 공유할 때의 오염을 ContextVar로 피한다.

**예상 결과:** 동시 요청2개의 trace_id가 서로 다르고 각 Trace에4개 Span. 무출력 통과.

**결과 해석:** 작업별 문맥을 분리해 다른 요청의 부모 ID가 섞이지 않는지 확인한다.

In [ ]:
async with lab_client(Settings(capacity=2)) as (app, client):
    responses = await asyncio.gather(*[client.post("/api/v1/assist", json={"message":str(i)}) for i in range(2)])
    snap = app.state.telemetry.snapshot()
tids = {r.headers["X-Trace-ID"] for r in responses}
assert len(tids) == 2
assert all(len([s for s in snap["spans"] if s["trace_id"] == tid]) == 4 for tid in tids)

## 7. 실패 지점

검색 실패 때 generate span이 생기지 않는 이유를 설명한다.

**예상 결과:** retrieve·assist ERROR, generate·format 없음.

**결과 해석:** HTTP502와 실행 구간을 함께 보면 검색 단계에서 중단됐음을 알 수 있다. 전체 시간이 길다는 설명보다 원인이 구체적이다.

In [ ]:
async with lab_client() as (app, client):
    await client.post("/api/v1/assist", json={"message":"예시","scenario":"retrieval_error"})
    spans = list(app.state.telemetry.spans)
assert not any(s["name"] == "generate" for s in spans)
[(s["name"],s["status"]) for s in spans]

## 직접 바꿔볼 실험 · 선택 복습

1. 정상·retrieval_error를 번갈아 실행해 생성된 Span 이름을 비교한다.
2. 시간 값 대신 부모·자식 포함 관계를 설명하는 간단한 도식을 그린다.
3. 학생 자신의 서비스에 필요한 Span 이름을3개 정의한다. 예: 문서 검색·재정렬·답변 생성.

한 번에 한 조건만 바꾸고 원래 코드와 비교한다. 기대값이 고정된 assert가 있다면 새 실험 조건에 맞는 값도 함께 설명한다.

## 프로젝트 적용 기록

**정리 질문:** 어느 단계가 실패하거나 느렸으며, 어떤 Trace·Span 관계가 그 판단을 뒷받침하는가?

아래 Markdown 셀을 더블클릭해 자신의 결과를 적는다. 다른 파일 제출이나 화면 실행을 자동으로 수행하지 않는다.

| 항목 | 직접 기록 |
|---|---|
| 예상 | 실행 전 예상한 결과 |
| 관찰 | 상태 코드·건수·실패 단계 등 실제 결과 |
| 근거 | 실행한 단계 번호·변경한 조건·캡처 위치 |
| 판단 | 유지·수정·채택·보류 이유 |
| 미검증·다음 변경 | 아직 실행하지 않은 조건과 다음 비교 |

학생 정보나 원문 입력은 공개 저장소에 올리지 않는다. 실제 실행하지 않은 조건은 미측정으로 표시한다.

## 막혔을 때의 확인 순서

| 증상 | 확인·해결 |
|---|---|
| 커널 선택 창만 표시 | Python·Jupyter 확장 설치 후 `.venv`의 Python 선택 |
| `ModuleNotFoundError` | 커널 경로 확인 → 해당 `.venv`로 `python -m pip install -r requirements.txt` |
| 환경 확인에서 저장소를 찾지 못함 | 저장소 전체를 열고 원래 `week10/lab/notebooks` 위치에서 실행 |
| `NameError` | 앞 셀 실행 여부 확인 → Restart Kernel → Run All |
| `asyncio.run() cannot be called...` | Jupyter에서는 제공된 최상위 `await`·`async with`를 그대로 사용 |
| 셀에 출력이 없음 | `assert`만 있는 셀은 오류 없이 끝나면 통과. 왼쪽 실행 번호와 완료 표시 확인 |
| 예상한 422·502·503·504 | 입력 오류·검색 오류·준비 실패·시간 초과를 의도적으로 만든 단계인지 확인 |
| 예상과 다른 `AssertionError` | 바로 위 코드·출력 확인. 변경한 값을 복원하고 처음부터 다시 실행. assert를 지워 통과시키지 않음 |
| 실행 시간·ID가 예시와 다름 | 고정값 비교 대신 상태·부모/자식 관계·분모·검증 결과를 비교 |

실습에서 바꾼 내용은 자신의 사본에 기록한다. 공개 저장소에는 학생 정보·원문 질문·실제 키를 올리지 않는다.